# Ch 7　執行模型內幕：Pregel、super-step 與 channels

> **本 notebook 完全不需要 API key。** 我們用可跑的小實驗，把「平行寫衝突」「平行讀不到」這些原本玄學的現象，眼睜睜看它發生、再修好。

In [ ]:
!uv pip install -q langgraph grandalf

## 7.5 坑一：兩個平行節點寫同一個 LastValue 欄位 → 必炸

`summary` 沒貼 reducer = `LastValue`，一步只能收一個值。兩個平行節點同時寫，引擎不知道留誰，直接報 `InvalidUpdateError`。

In [ ]:
from typing import Annotated, TypedDict
from operator import add
from langgraph.graph import StateGraph, START, END

class S1(TypedDict):
    summary: str       # 預設 LastValue：一步只能存一個值

def a(state): return {"summary": "from A"}
def b(state): return {"summary": "from B"}

g1 = StateGraph(S1)
g1.add_node("a", a); g1.add_node("b", b)
g1.add_edge(START, "a"); g1.add_edge(START, "b")    # 兩條都從 START 出發 = 同一 super-step 平行跑
g1.add_edge("a", END); g1.add_edge("b", END)
app1 = g1.compile()

try:
    app1.invoke({"summary": ""})
except Exception as e:
    print("如預期報錯 ->", type(e).__name__)
    print(str(e)[:140])
# InvalidUpdateError: At key 'summary': Can receive only one value per step. Use an Annotated key ...

## 解法：給這個欄位一個「能合併多筆寫入」的 reducer（= 換一種 channel）

In [ ]:
class S2(TypedDict):
    summary: Annotated[list[str], add]   # 改成可累加 -> 兩筆平行寫入會被合併

def a2(state): return {"summary": ["from A"]}
def b2(state): return {"summary": ["from B"]}

g2 = StateGraph(S2)
g2.add_node("a", a2); g2.add_node("b", b2)
g2.add_edge(START, "a"); g2.add_edge(START, "b")
g2.add_edge("a", END); g2.add_edge("b", END)
print(g2.compile().invoke({"summary": []}))
# {'summary': ['from A', 'from B']} —— 兩筆都留下了。一旦懂「Update 相位用 channel 的更新函數合併寫入」，這就理所當然。

## 7.4　Channels 就是 reducer 的真身

你在 Ch6 貼的 reducer，底層就是「channel 的更新函數」。直接玩玩 channel 你會更有感。

In [ ]:
import operator
from langgraph.channels import LastValue, BinaryOperatorAggregate

# LastValue：覆蓋（= Ch6 的預設 reducer）
last = LastValue(int)
# BinaryOperatorAggregate：用二元運算子累加（= Ch6 貼 operator.add 的真身）
total = BinaryOperatorAggregate(int, operator.add)
print("這兩個 channel，分別對應你在 Ch6 看到的『覆蓋』與『累加』。")
print("LastValue ->", type(last).__name__, "| BinaryOperatorAggregate ->", type(total).__name__)

## 7.2　數 super-step：平行的同一步，依序的不同步

下面這張圖：START 同時觸發 a、b（同一 super-step），兩者都寫進 `log`（用累加 reducer），再各自到 END。

In [ ]:
class S3(TypedDict):
    log: Annotated[list[str], add]

def step_a(state): return {"log": ["a 在 super-step 1 執行"]}
def step_b(state): return {"log": ["b 也在 super-step 1（和 a 平行）"]}

g3 = StateGraph(S3)
g3.add_node("a", step_a); g3.add_node("b", step_b)
g3.add_edge(START, "a"); g3.add_edge(START, "b")
g3.add_edge("a", END); g3.add_edge("b", END)

# stream(updates) 會把「同一步平行完成的節點」一起吐出來——這就是 super-step 的視覺證據。
for step in g3.compile().stream({"log": []}, stream_mode="updates"):
    print(step)

## 重點回顧

`StateGraph` 編譯出來的是一個 **Pregel** runtime；它用 super-step（平行同步、依序分步）推進，每步走 Plan→Execution→Update 三相，而 reducer 就是 channel 的更新函數。懂這層，平行衝突、平行讀不到、checkpoint 膨脹都不再是玄學——這也是你日後能拆開 Deep Agents 除錯的底氣。

## 拓樸圖：超步平行的圖長什麼樣

In [ ]:
# g3 = 7.2 那個「兩個平行節點 + START 同時觸發」的圖
print("--- ASCII ---")
print(g3.compile().get_graph().draw_ascii())
print("\n--- Mermaid 原始碼 ---")
print(g3.compile().get_graph().draw_mermaid())